In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

runs   = pd.read_csv("tpr_runs.csv",      encoding="utf-8-sig")
attack = pd.read_csv("attack_counts.csv", encoding="utf-8-sig")


def condition(model, input_type):
    """variant -> TPR array, ordered by run."""
    block = runs[
        (runs["model"] == model)
        & (runs["input_type"] == input_type)
        & (runs["attack_variant"] != "none")
    ]
    return {
        variant: group.sort_values("run")["tpr"].to_numpy(dtype=float)
        for variant, group in block.groupby("attack_variant")
    }


N = int(runs["n_malicious"].iloc[0])

tpr_orig_padded = condition("original",    "padded")
tpr_at_padded   = condition("adversarial", "padded")
tpr_at_unmod    = condition("adversarial", "unmodified")

tpr_orig_unmod = (
    runs[(runs["model"] == "original") & (runs["input_type"] == "unmodified")]
    .sort_values("run")["tpr"]
    .to_numpy(dtype=float)
)

B_c   = dict(zip(attack["attack_variant"], attack["benign_correct"].astype(int)))
evade = dict(zip(attack["attack_variant"], attack["successful_evasions"].astype(int)))

In [5]:
def ci(x):
    x = np.array(x); m = x.mean()
    h = stats.t.ppf(0.975, len(x)-1) * x.std(ddof=1)/np.sqrt(len(x))
    return m, h

oc, _ = ci(tpr_orig_unmod)
for c in ['std', 'comb', 'third']:
    p, ph = ci(tpr_orig_padded[c])
    a, ah = ci(tpr_at_padded[c])
    u, uh = ci(tpr_at_unmod[c])
    fn = (1 - p) * N  
    print(f'{c}: padded TPR {100*p:.2f}±{100*ph:.2f}  FN={fn:.1f}  evasions={evade[c]}')
    assert evade[c] <= fn + 0.5,  'FAIL: evasions > false negatives'
    assert abs((1-p)*100 - 100*evade[c]/N) > 0.05, 'WARN: TPR looks derived from ASR'
    print(f'   deg={100*(oc-p):.2f}  rec={100*(a-p):.2f}±{100*np.hypot(ah,ph):.2f}  '
          f'frac={100*(a-p)/(oc-p):.2f}%  unmod cost={100*(oc-u):.2f}')

std: padded TPR 83.18±0.70  FN=18.0  evasions=16
   deg=15.79  rec=12.15±0.89  frac=76.93%  unmod cost=0.84
comb: padded TPR 85.98±0.77  FN=15.0  evasions=13
   deg=12.99  rec=10.28±0.95  frac=79.14%  unmod cost=1.03
third: padded TPR 89.72±0.63  FN=11.0  evasions=9
   deg=9.25  rec=7.48±0.77  frac=80.81%  unmod cost=0.75
